In [11]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [12]:
import pandas as pd

skills_df = pd.read_csv("data/processed/skills.csv")

# Expect columns like: skill_id, skill_name
skills_df.head()


,skill_id,name,aliases,category,type
0,SK001,java,java,backend,technical
1,SK002,javascript,javascript,frontend,technical
2,SK003,sql,sql,database,technical
3,SK004,python,python,other,technical
4,SK005,css,css,frontend,technical


In [13]:
SKILL_ID_TO_NAME = dict(
    zip(skills_df["skill_id"], skills_df["name"])
)

def resolve_skill_names(skill_ids):
    return [
        SKILL_ID_TO_NAME.get(sid, sid)
        for sid in skill_ids
    ]


In [14]:
resolve_skill_names(["SK031", "SK004"])
# → ["Machine Learning", "Python"]


['machine learning', 'python']

In [15]:
def build_explanation_context(
    domain,
    current_role,
    next_role,
    readiness_score,
    missing_skills,
    matched_skills
):
    return {
        "domain": domain.replace("_", " "),
        "current_role": current_role.replace("_", " "),
        "next_role": next_role.replace("_", " "),
        "readiness_percentage": int(readiness_score * 100),
        "missing_skill_names": resolve_skill_names(missing_skills),
        "matched_skill_names": resolve_skill_names(matched_skills)
    }


In [16]:
def build_xai_prompt(context):
    return f"""
You are an AI career advisor.

Generate a clear, professional explanation for a user based ONLY on the information below.
Do not invent skills or roles.

Context:
- Domain: {context['domain']}
- Current role: {context['current_role']}
- Target role: {context['next_role']}
- Readiness: {context['readiness_percentage']}%
- Skills the user already has: {', '.join(context['matched_skill_names']) or 'None'}
- Skills the user is missing: {', '.join(context['missing_skill_names']) or 'None'}

Explain:
1. Why this role is the next logical step
2. What the readiness score means
3. What skills the user should focus on next

Use a friendly, human tone.
"""


In [19]:
def generate_dynamic_explanation(context):
    parts = []

    parts.append(
        f"As a {context['current_role']}, moving toward the role of "
        f"{context['next_role']} is a logical progression in the {context['domain']} field."
    )

    parts.append(
        f"Your current readiness is estimated at {context['readiness_percentage']}%, "
        f"based on how closely your existing skills align with job market expectations."
    )

    if context["missing_skill_names"]:
        parts.append(
            "To increase your readiness, it would be beneficial to focus on developing "
            + ", ".join(context["missing_skill_names"])
            + "."
        )

    if context["matched_skill_names"]:
        parts.append(
            "You already demonstrate strengths in "
            + ", ".join(context["matched_skill_names"])
            + ", which supports this career transition."
        )

    return " ".join(parts)


In [21]:
context = build_explanation_context(
 domain="SOFTWARE_ENGINEERING",
    current_role="JR_SE",
    next_role="JR_FE_DEV",
    readiness_score=0.55,
    missing_skills=["SK002", "SK005", "SK022"],   # javascript, css, html5
    matched_skills=["SK001", "SK003", "SK004"]   # java, sql, python
)

generate_dynamic_explanation(context)


'As a JR SE, moving toward the role of JR FE DEV is a logical progression in the SOFTWARE ENGINEERING field. Your current readiness is estimated at 55%, based on how closely your existing skills align with job market expectations. To increase your readiness, it would be beneficial to focus on developing javascript, css, html5. You already demonstrate strengths in java, sql, python, which supports this career transition.'